# 01 · The taxi simulator

`taxidemo` is a pure-Python, seedable port of the
[Emukit Playground](https://github.com/amzn/emukit-playground) taxi simulator —
a small agent-based model we use as a stand-in for POLARIS when demonstrating
`polarisopt` workflows. Everything that takes hours for a real DTA model runs in
under a second here, but the structure is the same: a stochastic black-box
simulator with a handful of tunable inputs and a few scalar outputs.

**The model.** A square grid of one-way roads. Taxis drive one tile per step.
Journey requests appear on sidewalks at a fixed rate, are dispatched to the
nearest available taxi, and pay
`base_fare + duration · (1 + q²·(max_multiplier − 1)) · cost_per_tile/10`
on completion (`q` = current demand backlog pressure). Every step also costs
`0.1 · taxi_count` to operate the fleet, and customers may refuse to book when
prices are set high.

| input | default | range | meaning |
|---|---|---|---|
| `grid_size` | 5 | 3–10 | roads per direction |
| `taxi_count` | 20 | 1–100 | fleet size |
| `journey_frequency` | 50 | 1–100 | requests per 10 steps |
| `base_fare` | 5 | 0–15 | flag-drop fare |
| `cost_per_tile` | 5 | 1–20 | per-km price |
| `max_multiplier` | 2 | 1–3 | surge-pricing cap |
| `max_steps` | 1000 | — | episode length |

| output | meaning |
|---|---|
| `profit` | total fares − operating cost |
| `missed` | customers lost (price-cancelled or never served) |
| `pick_up_time` | mean wait between request and pickup (steps) |
| `journeys_completed` | rides finished |
| `profit_per_journey` | gross fares / rides |


In [ ]:
from taxidemo import run_taxi_simulation, default_params

default_params()

## A single run

One call = one simulated episode. The `seed` argument makes runs reproducible —
the same seed always returns the same outputs.

In [ ]:
run_taxi_simulation(seed=0)

## The simulator is noisy

Like POLARIS, the model is stochastic: different seeds give different outcomes
for identical inputs. Any calibration or optimization on top of it has to deal
with this noise — which is why the `polarisopt` plugin averages `n_repeats`
seeded runs per design.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

profits = [run_taxi_simulation(seed=s)["profit"] for s in range(30)]
plt.figure(figsize=(6, 3.5))
plt.hist(profits, bins=12, edgecolor="k")
plt.xlabel("profit")
plt.ylabel("runs")
plt.title(f"30 seeds at default inputs — std ≈ {np.std(profits):,.0f}");

## Fleet-size trade-off

The playground's Bayesian-optimization tutorial optimizes `profit` over
`taxi_count`. More taxis serve more journeys and cut waiting times, but each
costs 0.1/step to run — so profit peaks somewhere in the middle. This is the
1-D slice of the response surface that workflow 3 (BO on Crossover) explores in
4-D.

In [ ]:
taxi_counts = np.arange(2, 101, 4)
runs = {tc: [run_taxi_simulation(seed=s, taxi_count=int(tc)) for s in range(3)] for tc in taxi_counts}

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, key in zip(axes, ["profit", "missed", "pick_up_time"]):
    ax.plot(taxi_counts, [np.mean([r[key] for r in runs[tc]]) for tc in taxi_counts], "o-")
    ax.set_xlabel("taxi_count")
    ax.set_ylabel(key)
fig.suptitle("Sweep of taxi_count (mean of 3 seeds, other inputs at defaults)")
fig.tight_layout();

## Pricing and lost customers

The playground's sensitivity-analysis tutorial varies `cost_per_tile`. Past the
midpoint of the price slider customers start refusing to book, so revenue first
rises with price and then collapses.

In [ ]:
prices = np.linspace(1, 20, 20)
runs = {c: [run_taxi_simulation(seed=s, cost_per_tile=float(c)) for s in range(3)] for c in prices}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, key in zip(axes, ["profit", "missed"]):
    ax.plot(prices, [np.mean([r[key] for r in runs[c]]) for c in prices], "o-")
    ax.set_xlabel("cost_per_tile")
    ax.set_ylabel(key)
fig.suptitle("Sweep of cost_per_tile (mean of 3 seeds)")
fig.tight_layout();

## Next

- **[02_lhs_local.ipynb](02_lhs_local.ipynb)** — screen all four pricing/fleet
  knobs at once with a Latin-hypercube study run by `polarisopt` on this
  machine.
- **[03_bo_crossover.ipynb](03_bo_crossover.ipynb)** — analyze the Bayesian
  optimization study run on the LCRC Crossover cluster via the `polarisopt`
  CLI.